# RAG: AI tumhari files se jawab kaise deta hai?
**AI Explained · Episode 4 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai, kuch install nahi karna.

Hum ek **mini RAG** banayenge, ek nakli company ki handbook par. Asli RAG mein meaning ke liye embedding model aur jawab ke liye LLM hota hai; yahan dono ka simple version hai, taaki har step dikhe.

In [1]:
handbook = """
Leave: Har employee ko saal mein 24 leave
milti hain. Leave se 3 din pehle manager
ko batana hota hai.

WFH: Hafte mein 2 din work from home
allowed hai.

Salary: Salary har mahine ki 1 tareekh
ko account mein aati hai.

Office: Office ka time subah 10 se
shaam 6 baje tak hai.

Laptop: Naya laptop IT team se ticket
daal ke milta hai.
"""
print(len(handbook.split()), "words")

65 words


In [2]:
chunks = [c.replace("\n", " ").strip()
          for c in handbook.split("\n\n")]
for i, c in enumerate(chunks, 1):
    print(i, c[:40] + "...")

1 Leave: Har employee ko saal mein 24 leav...
2 WFH: Hafte mein 2 din work from home all...
3 Salary: Salary har mahine ki 1 tareekh k...
4 Office: Office ka time subah 10 se shaam...
5 Laptop: Naya laptop IT team se ticket da...


In [3]:
import math, re
from collections import Counter

def vec(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return Counter(words)

print(vec("Salary kab aati hai? Salary!"))

Counter({'salary': 2, 'kab': 1, 'aati': 1, 'hai': 1})


In [4]:
def cosine(a, b):
    dot = sum(a[w] * b[w] for w in a)
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0

vecs = [vec(c) for c in chunks]
print(round(cosine(vec("leave"), vecs[0]), 2))

0.57


In [5]:
def search(q, k=2):
    s = [(round(cosine(vec(q), v), 2), i + 1)
         for i, v in enumerate(vecs)]
    return sorted(s, reverse=True)[:k]

print(search("Saal mein kitni leave milti hain?"))

[(0.54, 1), (0.13, 2)]


In [6]:
print(search("Paisa kab aata hai?"))

[(0.16, 2), (0.14, 5)]


In [7]:
# Embedding model ye khud seekhta hai.
# Yahan 4 words haath se:
SAME = {"paisa": "salary", "chutti": "leave",
        "ghar": "wfh", "aata": "aati"}

def vec(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    return Counter(SAME.get(w, w) for w in words)

vecs = [vec(c) for c in chunks]
print(search("Paisa kab aata hai?"))

[(0.53, 3), (0.16, 2)]


In [8]:
def make_prompt(q):
    top = search(q)
    ctx = "\n".join(f"[{i}] {chunks[i - 1]}"
                    for s, i in top if s > 0)
    return ("Sirf is context se jawab do. Na mile "
            "to 'pata nahi' bolo.\n\n" + ctx +
            "\n\nSawal: " + q)

print(make_prompt("Mujhe kitni chutti milti hai?"))

Sirf is context se jawab do. Na mile to 'pata nahi' bolo.

[1] Leave: Har employee ko saal mein 24 leave milti hain. Leave se 3 din pehle manager ko batana hota hai.
[2] WFH: Hafte mein 2 din work from home allowed hai.

Sawal: Mujhe kitni chutti milti hai?


In [9]:
def answer(q):
    (score, i), = search(q, k=1)
    if score < 0.2:
        return "Pata nahi (handbook mein nahi hai)"
    first = chunks[i - 1].split(". ")[0]
    first = first.rstrip(".")
    return f"{first}.\n[source {i}]"

print(answer("Mujhe kitni chutti milti hai?"))
print(answer("Ghar se kaam kitne din?"))

Leave: Har employee ko saal mein 24 leave milti hain.
[source 1]
WFH: Hafte mein 2 din work from home allowed hai.
[source 2]


In [10]:
print(search("Company ke CEO kaun hain?"))

[(0.12, 5), (0.08, 1)]


In [11]:
print(answer("Company ke CEO kaun hain?"))

Pata nahi (handbook mein nahi hai)


In [12]:
# Apna sawal poochho:
q = "Office kitne baje khulta hai?"
print(search(q))
print(answer(q))

[(0.48, 4), (0.14, 2)]
Office: Office ka time subah 10 se shaam 6 baje tak hai.
[source 4]
